## Gaussian Segmentation Network with Banded Covariance on LIDC-IDRI

Train and evaluate a **Gaussian Segmentation Network** with banded covariance structure.
Samples are drawn from a learned conditional Gaussian with spatial correlations and passed through softmax to obtain class probabilities.

**No flow augmentation** - direct prediction from the base distribution.

In [1]:
import pickle
import argparse

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

### Configuration & hyperparameters

In [2]:
res = 128
num_classes = 2

args = argparse.Namespace(
    data_dir="/Users/vzer/projects/flow-ssn/lidc/data_lidc.hdf5",
    resolution=res,
    img_channels=1,
    num_classes=num_classes,
    model="gauss",
    base_net="unet",
    cond_base=True,
    base_std=1.0,
    band_width=1,  # banded covariance: 0=diagonal, 1=3x3, 2=5x5, etc.
    eval_samples=32,
    seed=8,
    # base unet
    base_input_shape=[1, res, res],
    base_model_channels=32,
    base_out_channels=num_classes * 2,  # loc + log_scale
    base_num_res_blocks=1,
    base_attention_resolutions=[0],
    base_dropout=0.1,
    base_channel_mult=[1, 2, 4, 8],
    base_num_heads=1,
    base_num_head_channels=64,
)
print(f"Config: band_width={args.band_width}, cond_base={args.cond_base}")

Config: band_width=1, cond_base=True


### Load LIDC-IDRI dataset

In [3]:
from gssn.data.lidc import get_lidc, make_dataloader, preprocess_lidc_fn

datasets = get_lidc(args)
test_rng = jax.random.PRNGKey(42)
batch_size = 12

print(f"Train: {len(datasets['train'])}, Valid: {len(datasets['valid'])}, Test: {len(datasets['test'])}")

Loading LIDC train:
images: (9541, 1, 128, 128), masks: (9541, 4, 128, 128)
Loading LIDC val:
images: (2482, 1, 128, 128), masks: (2482, 4, 128, 128)
Loading LIDC test:
images: (3073, 1, 128, 128), masks: (3073, 4, 128, 128)
Train: 9541, Valid: 2482, Test: 3073


### Build model

In [4]:
from gssn.factory import build_nn
from gssn.utils import count_params
from gssn.models.gnn.model import GaussianSegmentationNetwork

# Build base network (UNet)
base_net = None
if args.base_net != "" and args.cond_base:
    base_net, _ = build_nn(args.base_net, args=args, prefix="base_")

# Create Gaussian Segmentation Network
model = GaussianSegmentationNetwork(
    base_net=base_net,
    num_classes=args.num_classes,
    cond_base=args.cond_base,
    base_std=args.base_std,
    band_width=args.band_width,
)

# Initialize params
dummy_batch = {"x": jnp.zeros((1, 1, 128, 128))}
init_rng = jax.random.PRNGKey(0)
params = model.init({"params": init_rng, "sample": init_rng}, dummy_batch)
print("Initialized model params")

p = params.get("params", params)
base_count = count_params(p["base_net"]) if "base_net" in p else 0
corr_count = count_params(p["correlation_kernel"]) if "correlation_kernel" in p else 0
print(
    f"#base_net params: {base_count:,}\n"
    f"#correlation_kernel params: {corr_count:,}\n"
    f"#total params: {count_params(params):,}"
)

Initialized model params
#base_net params: 13,591,748
#correlation_kernel params: 18
#total params: 13,591,766


### Training setup

In [5]:
import functools
import optax
from flax.training import train_state
from gssn.utils import EMA, create_lr_schedule
from gssn.data.lidc import augment_lidc_batch

# --- Hyperparameters ---
lr = 1e-4
lr_warmup = 2000
weight_decay = 1e-4
ema_rate = 0.9999
mc_samples_train = 1
train_bs = 32
num_epochs = 1001
eval_freq = 16
eval_samples = 32

# --- Optimizer (adamw + linear warmup + grad clip) ---
lr_schedule = create_lr_schedule(lr, lr_warmup, total_steps=0)
tx = optax.chain(
    optax.clip_by_global_norm(10.0),
    optax.adamw(lr_schedule, weight_decay=weight_decay),
)

# --- TrainState bundles params + optimizer + apply_fn ---
state = train_state.TrainState.create(
    apply_fn=model.apply,
    params=params["params"],
    tx=tx,
)

# --- EMA ---
ema = EMA({"params": state.params}, rate=ema_rate)

print(f"Optimizer initialized | lr={lr}, warmup={lr_warmup}, wd={weight_decay}")
print(f"Training: {num_epochs} epochs, bs={train_bs}, mc={mc_samples_train}")
print(f"Eval: every {eval_freq} epochs, {eval_samples} samples")

Optimizer initialized | lr=0.0001, warmup=2000, wd=0.0001
Training: 1001 epochs, bs=32, mc=1
Eval: every 16 epochs, 32 samples


### Training step (JIT-compiled)

In [6]:
@functools.partial(jax.jit, static_argnames=("apply_fn",))
def train_step(state, batch, rng, apply_fn):
    """JIT-compiled training step. apply_fn is static to avoid closure issues."""
    def loss_fn(params):
        out = apply_fn(
            {"params": params},
            batch,
            mc_samples=1,
            rng=rng,
            deterministic=False,
            rngs={"sample": rng, "dropout": jax.random.fold_in(rng, 1)},
        )
        return out["loss"], out["std"]

    (loss, std), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    grad_norm = optax.global_norm(grads)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss, std, grad_norm

print("train_step defined (will JIT on first call)")

train_step defined (will JIT on first call)


### Evaluation function

In [7]:
from gssn.eval.metrics import energy_distance, hungarian_matched_iou, dice_score

@jax.jit
def eval_batch(params, apply_fn, batch, rng, mc_samples):
    """Evaluate one batch with mc_samples."""
    out = apply_fn(
        params,
        batch,
        mc_samples=mc_samples,
        rng=rng,
        deterministic=True,
        rngs={"sample": rng},
    )
    return out["probs"]  # (mc, B, H, W, K)


def run_eval_epoch(model, params, dataset, rng, batch_size, eval_samples, num_classes):
    """Run evaluation over entire dataset."""
    keys = ["energy_distance", "diversity", "hmiou", "dice"]
    metrics = {k: 0.0 for k in keys}
    counts = {k: 0.0 for k in keys}

    for batch in make_dataloader(dataset, batch_size, rng, shuffle=False):
        rng, rng_step = jax.random.split(rng)
        batch = preprocess_lidc_fn(batch, rng=rng_step)

        rng, rng_eval = jax.random.split(rng)
        probs = eval_batch({"params": params}, model.apply, batch, rng_eval, eval_samples)

        # One-hot predictions
        pred_oh = jax.nn.one_hot(jnp.argmax(probs, axis=-1), num_classes)

        # Ground truth
        y_all = batch["y_all"]  # (B, H, W, R)
        r = y_all.shape[-1]
        gt_oh = jax.nn.one_hot(y_all.astype(jnp.int32), num_classes)
        gt_oh = jnp.transpose(gt_oh, (3, 0, 1, 2, 4))  # (R, B, H, W, K)

        # Metrics
        ged, div = energy_distance(pred_oh, gt_oh)
        hm = hungarian_matched_iou(pred_oh[:r], gt_oh)

        # Dice
        mean_pred = jnp.mean(probs, axis=0)
        mean_pred_oh = jax.nn.one_hot(jnp.argmax(mean_pred, axis=-1), num_classes)
        d_scores = []
        for j in range(r):
            d_scores.append(dice_score(mean_pred_oh, gt_oh[j]))
        d_arr = jnp.stack(d_scores, axis=0).mean(axis=0)

        for k, v in [("energy_distance", ged), ("diversity", div), ("hmiou", hm), ("dice", d_arr)]:
            valid = v[~jnp.isnan(v)]
            metrics[k] += float(jnp.sum(valid))
            counts[k] += int(valid.size)

    return {k: metrics[k] / counts[k] if counts[k] > 0 else 0.0 for k in keys}

print("Evaluation functions defined")

Evaluation functions defined


### Training loop

In [8]:
rng = jax.random.PRNGKey(args.seed)
best_ged = float("inf")
best_dice = 0.0

for epoch in range(num_epochs):
    rng, rng_epoch = jax.random.split(rng)
    rng_loader, rng_train = jax.random.split(rng_epoch)

    loader = make_dataloader(
        datasets["train"], train_bs, rng_loader, shuffle=True, drop_last=True,
    )
    batches = list(loader)
    pbar = tqdm(batches, total=len(batches), mininterval=1.0)

    epoch_loss, epoch_count = 0.0, 0

    for batch in pbar:
        rng_train, rng_step = jax.random.split(rng_train)
        rng_aug, rng_pre, rng_model = jax.random.split(rng_step, 3)

        # Augment + preprocess
        batch = augment_lidc_batch(batch, rng_aug)
        batch = preprocess_lidc_fn(batch, rng=rng_pre)
        bs = batch["x"].shape[0]

        state, loss, std, grad_norm = train_step(state, batch, rng_model, model.apply)
        ema.update({"params": state.params})

        epoch_loss += float(loss) * bs
        epoch_count += bs
        avg = epoch_loss / epoch_count
        pbar.set_description(
            f"[epoch {epoch}] loss: {avg:.4f}, std: {float(std):.1e}, gnorm: {float(grad_norm):.2f}"
        )

    # --- Periodic evaluation ---
    if epoch > 0 and (epoch % eval_freq) == 0:
        rng, rng_eval = jax.random.split(rng)
        eval_params = ema.get()

        valid_metrics = run_eval_epoch(
            model, eval_params, datasets["valid"], rng_eval,
            batch_size=train_bs,
            eval_samples=eval_samples,
            num_classes=args.num_classes,
        )
        print(
            f"\n  [epoch {epoch}] train_loss: {epoch_loss / epoch_count:.5f}"
            f"\n  valid: {', '.join(f'{k}: {v:.5f}' for k, v in valid_metrics.items())}"
        )

        # Track best
        ged = valid_metrics.get("energy_distance", float("inf"))
        dice = valid_metrics.get("dice", 0.0)
        if ged < best_ged:
            best_ged = ged
            best_params_ged = jax.device_get(eval_params)
            print(f"  ** new best GED: {best_ged:.5f}")
        if dice > best_dice:
            best_dice = dice
            best_params_dice = jax.device_get(eval_params)
            print(f"  ** new best Dice: {best_dice:.5f}")

# Store final params for downstream cells
params = ema.get()
print(f"\nTraining complete. Best GED: {best_ged:.5f}, Best Dice: {best_dice:.5f}")

[epoch 0] loss: 0.7196, std: 0.0e+00, gnorm: 6.48:  20%|██        | 60/298 [11:34<45:54, 11.57s/it] 


KeyboardInterrupt: 

### Test evaluation

In [ ]:
mc_samples = 100

keys = ["energy_distance", "diversity", "hmiou", "dice"]
metrics = {k: 0.0 for k in keys}
counts = {k: 0.0 for k in keys}

rng_eval = jax.random.PRNGKey(0)

for batch in tqdm(
    make_dataloader(datasets["test"], batch_size, test_rng, shuffle=False),
    total=(len(datasets["test"]) + batch_size - 1) // batch_size,
    desc="[test evaluation]",
):
    rng_eval, rng_step = jax.random.split(rng_eval)
    batch = preprocess_lidc_fn(batch, rng=rng_step)

    rng_eval, rng_sample = jax.random.split(rng_eval)
    probs = eval_batch({"params": params}, model.apply, batch, rng_sample, mc_samples)

    # One-hot predictions: (mc, B, H, W, K)
    pred_oh = jax.nn.one_hot(jnp.argmax(probs, axis=-1), model.num_classes)

    # Ground truth raters: (R, B, H, W, K)
    y_all = batch["y_all"]  # (B, H, W, R)
    r = y_all.shape[-1]
    gt_oh = jax.nn.one_hot(y_all.astype(jnp.int32), model.num_classes)
    gt_oh = jnp.transpose(gt_oh, (3, 0, 1, 2, 4))  # (R, B, H, W, K)

    # --- Metrics ---
    ged, div = energy_distance(pred_oh, gt_oh)
    hm = hungarian_matched_iou(pred_oh[:r], gt_oh)

    # Dice: mean prediction vs each GT rater, then average
    mean_pred = jnp.mean(probs, axis=0)
    mean_pred_oh = jax.nn.one_hot(jnp.argmax(mean_pred, axis=-1), model.num_classes)
    d_scores = []
    for j in range(r):
        d_scores.append(dice_score(mean_pred_oh, gt_oh[j]))
    d_arr = jnp.stack(d_scores, axis=0).mean(axis=0)

    for k, v in [("energy_distance", ged), ("diversity", div), ("hmiou", hm), ("dice", d_arr)]:
        valid = v[~jnp.isnan(v)]
        metrics[k] += float(jnp.sum(valid))
        counts[k] += int(valid.size)

print("\n--- Test results ---")
for k in keys:
    if counts[k] > 0:
        print(f"  {k}: {metrics[k] / counts[k]:.4f}")

### Plot qualitative results

In [ ]:
taborange = (1.0, 127 / 255, 14 / 255)
skyblue = (0.529, 0.808, 0.922)
light_sea_green = (32 / 255, 178 / 255, 170 / 255)


def get_overlay(
    mask: np.ndarray,
    img: np.ndarray,
    rgb: tuple,
    alpha: float = 0.6,
) -> np.ndarray:
    """Overlay a binary mask on an image with a given color."""
    color_mask = np.zeros_like(img)
    for i in range(3):
        color_mask[..., i] = mask * rgb[i]
    overlay = img.copy()
    nonzero = mask > 0
    overlay[nonzero] = (1 - alpha) * img[nonzero] + alpha * color_mask[nonzero]
    return np.clip(overlay, 0, 1)

In [ ]:
# Grab a fresh batch
plot_rng = jax.random.PRNGKey(123)
plot_batch = next(make_dataloader(datasets["test"], batch_size, plot_rng, shuffle=True))
plot_batch = preprocess_lidc_fn(plot_batch, rng=plot_rng)

_, rng_plot_sample = jax.random.split(plot_rng)
plot_probs = eval_batch({"params": params}, model.apply, plot_batch, rng_plot_sample, mc_samples)

for _ in range(3):
    b = np.random.randint(plot_batch["x"].shape[0])
    n = 10
    fig, ax = plt.subplots(1, n, figsize=(n * 5 - 5, 5))

    # Image: NCHW -> HWC, rescale [-1,1] -> [0,1], crop center, tile to RGB
    img_np = np.array(plot_batch["x"][b]).transpose(1, 2, 0)
    img_np = (img_np + 1) * 0.5
    o = 32
    img_np = img_np[o:128 - o, o:128 - o]
    img_np = np.repeat(img_np, 3, axis=-1)  # grayscale -> RGB

    # 4 GT rater labels
    y_all_np = np.array(plot_batch["y_all"][b])  # (H, W, R)
    for i in range(4):
        mask = y_all_np[o:128 - o, o:128 - o, i]
        overlay = get_overlay(mask, img_np, skyblue)
        ax[i].imshow(overlay)
        ax[i].set_title(f"Label {i + 1}", fontsize=20, pad=10)
        ax[i].axis("off")

    # 4 predicted samples
    preds_np = np.array(plot_probs[:, b])  # (mc, H, W, K)
    for i in range(4):
        mask = preds_np[i, o:128 - o, o:128 - o].argmax(-1)
        overlay = get_overlay(mask, img_np, taborange)
        ax[4 + i].imshow(overlay)
        ax[4 + i].set_title(f"Sample {i + 1}", fontsize=20, pad=10)
        ax[4 + i].axis("off")

    # Mean prediction
    mean_pred_np = preds_np.mean(axis=0)  # (H, W, K)
    mean_mask = mean_pred_np[o:128 - o, o:128 - o].argmax(-1)
    overlay = get_overlay(mean_mask, img_np, light_sea_green)
    ax[-2].imshow(overlay)
    ax[-2].set_title("Mean", fontsize=20, pad=10)
    ax[-2].axis("off")

    # Uncertainty (entropy)
    entropy = -np.sum(mean_pred_np * np.log(mean_pred_np + 1e-12), axis=-1)
    entropy = entropy[o:128 - o, o:128 - o]
    entropy_norm = entropy / (entropy.max() + 1e-12)
    ax[-1].imshow(entropy_norm, cmap="Reds", vmin=0, vmax=1)
    ax[-1].set_title("Uncertainty", fontsize=20, pad=10)
    ax[-1].axis("off")

    plt.tight_layout()
    plt.show()